In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path
import tarfile, os

archive = Path("/content/drive/MyDrive/stargan-v2/afhq_all-20250824-0531.tar.gz")
dest    = Path("/content/stargan-v2")
dest.mkdir(parents=True, exist_ok=True)

# Extract the .tar.gz
with tarfile.open(archive, mode="r:gz") as tf:
    tf.extractall(path=dest)  # contents go under /content/stargan-v2/...

# Sanity checks (size + file count)
data_root = dest / "data/afhq_all"

# Count files
file_count = sum(1 for p in data_root.rglob("*") if p.is_file())

# Folder size
total_bytes = sum(p.stat().st_size for p in data_root.rglob("*") if p.is_file())
def hr(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

print("Extracted to:", data_root)
print("Files:", file_count)
print("Size:", hr(total_bytes))

# Peek a few files
print("Sample files:")
for p in list(data_root.rglob("*"))[:5]:
    if p.is_file():
        print(" -", p)


/tmp/ipython-input-3009054212.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(path=dest)  # contents go under /content/stargan-v2/...


Extracted to: /content/stargan-v2/data/afhq_all
Files: 16126
Size: 695.4 MB
Sample files:


In [ ]:
from pathlib import Path
import tarfile, os

archive = Path("/content/drive/MyDrive/stargan-v2/afhq_all-20250824-0532.tar.gz")
dest    = Path("/content/stargan-v2")
dest.mkdir(parents=True, exist_ok=True)

# Extract the .tar.gz
with tarfile.open(archive, mode="r:gz") as tf:
    tf.extractall(path=dest)  # contents go under /content/stargan-v2/...

# Sanity checks (size + file count)
data_root = dest / "data/afhq_all"

# Count files
file_count = sum(1 for p in data_root.rglob("*") if p.is_file())

# Folder size
total_bytes = sum(p.stat().st_size for p in data_root.rglob("*") if p.is_file())
def hr(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

print("Extracted to:", data_root)
print("Files:", file_count)
print("Size:", hr(total_bytes))

# Peek a few files
print("Sample files:")
for p in list(data_root.rglob("*"))[:5]:
    if p.is_file():
        print(" -", p)


/tmp/ipython-input-3331895069.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(path=dest)  # contents go under /content/stargan-v2/...


Extracted to: /content/stargan-v2/data/afhq_all
Files: 16126
Size: 695.4 MB
Sample files:


In [ ]:
# --- SETTINGS ---
IMG_SIZE = 256  # StarGAN v2 default; change if you sampled with another size

from pathlib import Path
from PIL import Image
import os, pandas as pd, re

base_drive = Path("/content/stargan-v2")
in_root    = base_drive/"expr/results/afhq_all_fake_stargan"
out_root   = base_drive/"expr/results/afhq_all_fake_tiles"
meta_path  = base_drive/"data/afhq_all_metadata.csv"
splits     = ["train","val","holdout"]

out_root.mkdir(parents=True, exist_ok=True)
new_rows = []

def slice_grid(ref_path: Path, split: str):
    img = Image.open(ref_path).convert("RGB")
    W, H = img.size
    cols = W // IMG_SIZE
    rows = H // IMG_SIZE
    if cols == 0 or rows == 0:
        print(f"[WARN] {ref_path} not divisible by {IMG_SIZE}; skipping.")
        return 0
    # Save tiles
    out_dir = out_root/split
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = 0
    for r in range(rows):
        for c in range(cols):
            # If you want to SKIP the first column (often the source image), uncomment:
            if c == 0: continue
            tile = img.crop((c*IMG_SIZE, r*IMG_SIZE, (c+1)*IMG_SIZE, (r+1)*IMG_SIZE))
            fname = f"{split}_r{r:04d}_c{c:04d}.png"
            fpath = out_dir/fname
            tile.save(fpath, "PNG")
            new_rows.append({
                "id": fpath.stem,
                "path": str(fpath),
                "split": split,
                "species_src": "",         # unknown from grid; leave blank
                "species_tgt": "",
                "label": "fake",
                "gen_family": "stargan_v2",
                "seed": "",
                "ref_id": "",
                "source": "generated_grid"
            })
            saved += 1
    return saved

total_saved = 0
for sp in splits:
    ref = in_root/f"{sp}_all"/"reference.jpg"
    if ref.exists():
        print(f"Slicing {ref} ...")
        total_saved += slice_grid(ref, sp)
    else:
        print(f"[INFO] No {ref} (did sampling finish for {sp}?)")

print(f"\nSaved {total_saved} tiles to {out_root}")

# Append to metadata (creating it if missing)
"""
if meta_path.exists():
    df = pd.read_csv(meta_path)
else:
    df = pd.DataFrame(columns=["id","path","split","species_src","species_tgt",
                               "label","gen_family","seed","ref_id","source"])
df_new = pd.DataFrame(new_rows)
df_all = pd.concat([df, df_new], ignore_index=True)
df_all.to_csv(meta_path, index=False)
print(f"Updated metadata: {meta_path} (rows={len(df_all)})")
"""

Slicing /content/stargan-v2/expr/results/afhq_all_fake_stargan/train_all/reference.jpg ...
Slicing /content/stargan-v2/expr/results/afhq_all_fake_stargan/val_all/reference.jpg ...
Slicing /content/stargan-v2/expr/results/afhq_all_fake_stargan/holdout_all/reference.jpg ...

Saved 2976 tiles to /content/stargan-v2/expr/results/afhq_all_fake_tiles
Updated metadata: /content/stargan-v2/data/afhq_all_metadata.csv (rows=2976)


In [ ]:
from pathlib import Path
import tarfile, os

archive = Path("/content/drive/MyDrive/stargan-v2/afhq_all-20250824-0541.tar.gz")
dest    = Path("/content/stargan-v2")
dest.mkdir(parents=True, exist_ok=True)

# Extract the .tar.gz
with tarfile.open(archive, mode="r:gz") as tf:
    tf.extractall(path=dest)  # contents go under /content/stargan-v2/...

# Sanity checks (size + file count)
data_root = dest / "data/afhq_all"

# Count files
file_count = sum(1 for p in data_root.rglob("*") if p.is_file())

# Folder size (human-readable)
total_bytes = sum(p.stat().st_size for p in data_root.rglob("*") if p.is_file())
def hr(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

print("Extracted to:", data_root)
print("Files:", file_count)
print("Size:", hr(total_bytes))

# Peek a few files
print("Sample files:")
for p in list(data_root.rglob("*"))[:5]:
    if p.is_file():
        print(" -", p)


/tmp/ipython-input-2479020740.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(path=dest)  # contents go under /content/stargan-v2/...


Extracted to: /content/stargan-v2/data/afhq_all
Files: 16126
Size: 695.4 MB
Sample files:


In [ ]:
# Colab sampler (robust to both layouts: with or without cat/dog/wild subfolders)
import os, random, hashlib, shutil
from pathlib import Path
from collections import defaultdict

SEED = 42
random.seed(SEED)

# Sources
REAL_ROOT = Path("/content/stargan-v2/data/afhq_all")
FAKE_TECH_ROOTS = {
    "stargan_tiles": Path("/content/stargan-v2/expr/results/afhq_all_fake_tiles"),
    "copy_move":     Path("/content/stargan-v2/expr/results/fake_copy_move"),
    "inpaint":       Path("/content/stargan-v2/expr/results/fake_inpaint"),
    "postproc":      Path("/content/stargan-v2/expr/results/fake_postproc"),
    "splicing":      Path("/content/stargan-v2/expr/results/fake_splicing"),
    "swap_like":     Path("/content/stargan-v2/expr/results/fake_swap_like"),
}

SPLITS  = ["train","val","holdout"]
SPECIES = ["cat","dog","wild"]
EXTS    = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

OUT_ROOT = Path("/content/stargan-v2/data/combined_balanced")
COPY_MODE = "copy"      # or "copy"
MAX_REALS_PER_SPLIT = None # e.g. 5000 to cap; None = no cap

def list_images_dir(d: Path):
    if not d.exists(): return []
    return [p for p in d.iterdir() if p.is_file() and p.suffix.lower() in EXTS]

def list_images_split_species_or_recursive(root: Path, split: str):
    """Prefer species subfolders; else recursively scan the split folder."""
    base = root / split
    if not base.exists(): return []
    has_species = any((base/s).is_dir() for s in SPECIES)
    if has_species:
        imgs = []
        for s in SPECIES:
            imgs += list_images_dir(base/s)
        return imgs, "species"
    else:
        return [p for p in base.rglob("*") if p.is_file() and p.suffix.lower() in EXTS], "recursive"

def list_real_split(root: Path, split: str):
    # reals are expected to have species subfolders; fall back to recursive if not
    imgs, mode = list_images_split_species_or_recursive(root, split)
    return imgs, mode

def sha1_file(fp: Path, chunk=1<<20):
    h = hashlib.sha1()
    with open(fp,"rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists(): return
    if COPY_MODE == "copy":
        shutil.copy2(src, dst)
    else:
        try: os.symlink(src, dst)
        except OSError: shutil.copy2(src, dst)

# 1) Scan sources (with layout detection)
real_by_split = {}
real_mode_by_split = {}
for sp in SPLITS:
    imgs, mode = list_real_split(REAL_ROOT, sp)
    real_by_split[sp] = imgs
    real_mode_by_split[sp] = mode

fake_by_split_by_tech = {sp:{} for sp in SPLITS}
fake_mode_by_split_by_tech = {sp:{} for sp in SPLITS}
for tech, root in FAKE_TECH_ROOTS.items():
    for sp in SPLITS:
        imgs, mode = list_images_split_species_or_recursive(root, sp)
        fake_by_split_by_tech[sp][tech] = imgs
        fake_mode_by_split_by_tech[sp][tech] = mode

print("Available images (found):")
for sp in SPLITS:
    fcounts = {t: len(fake_by_split_by_tech[sp][t]) for t in FAKE_TECH_ROOTS}
    print(f"- {sp}: real={len(real_by_split[sp])} (mode={real_mode_by_split[sp]}) | fakes by tech =", fcounts)
    layout_info = {t: fake_mode_by_split_by_tech[sp][t] for t in FAKE_TECH_ROOTS}
    print("  layout:", layout_info)

# 2) Decide sample sizes (1:1 real:fake, fakes balanced across techniques)
def decide_sample_sizes(split):
    n_real_avail = len(real_by_split[split])
    if MAX_REALS_PER_SPLIT is not None:
        n_real_avail = min(n_real_avail, MAX_REALS_PER_SPLIT)

    tech_avail = {t: len(fake_by_split_by_tech[split][t]) for t in FAKE_TECH_ROOTS}
    usable = [t for t,n in tech_avail.items() if n > 0]
    if not usable or n_real_avail == 0:
        return 0, {}

    total_fakes_avail = sum(tech_avail[t] for t in usable)
    target = min(n_real_avail, total_fakes_avail)

    per = {t:0 for t in usable}
    base = target // len(usable)
    for t in usable:
        per[t] = min(base, tech_avail[t])
    rem = target - sum(per.values())
    if rem > 0:
        # give remainder to techniques with most headroom
        for t in sorted(usable, key=lambda x: (tech_avail[x] - per[x]), reverse=True):
            if rem == 0: break
            if per[t] < tech_avail[t]:
                per[t] += 1
                rem -= 1

    final_fakes = sum(per.values())
    final_reals = min(n_real_avail, final_fakes)
    # Trim fakes to match if needed
    over = final_fakes - final_reals
    if over > 0:
        for t in sorted(per, key=lambda x: per[x], reverse=True):
            if over == 0: break
            take = min(over, per[t])
            per[t] -= take
            over -= take
        final_fakes = sum(per.values())
        final_reals = min(final_reals, final_fakes)
    return final_reals, per

# 3) Sample & write
OUT_ROOT.mkdir(parents=True, exist_ok=True)
for sp in SPLITS:
    out_real = OUT_ROOT/sp/"real"
    out_fake = OUT_ROOT/sp/"fake"
    out_real.mkdir(parents=True, exist_ok=True)
    out_fake.mkdir(parents=True, exist_ok=True)

    n_reals, per_tech = decide_sample_sizes(sp)
    if n_reals == 0:
        print(f"[{sp}] No usable data; skipping.")
        continue

    reals = real_by_split[sp][:]
    random.shuffle(reals)
    reals = reals[:n_reals]

    fake_samples = []
    for t, k in per_tech.items():
        if k <= 0: continue
        pool = fake_by_split_by_tech[sp][t][:]
        random.shuffle(pool)
        fake_samples += [(p, t) for p in pool[:k]]

    # Align counts
    if len(fake_samples) > len(reals):
        fake_samples = fake_samples[:len(reals)]
    elif len(fake_samples) < len(reals):
        reals = reals[:len(fake_samples)]

    # Write
    for p in reals:
        link_or_copy(p, out_real/p.name)
    for p, tech in fake_samples:
        link_or_copy(p, out_fake/f"{p.stem}__{tech}{p.suffix.lower()}")

    print(f"[{sp}] wrote real={len(reals)} fake={len(fake_samples)} per-tech={per_tech}")

# 4) Counts
def count_files(d: Path): return sum(1 for x in d.glob("*") if x.is_file())
print("\nFinal counts in:", OUT_ROOT)
for sp in SPLITS:
    r = count_files(OUT_ROOT/sp/"real")
    f = count_files(OUT_ROOT/sp/"fake")
    print(f"- {sp:7s} real={r:5d}  fake={f:5d}  total={r+f}")

# 5) Leakage checks (cross-split; and within-split real vs fake)
def collect_hashes(paths):
    h2p = defaultdict(list)
    for p in paths:
        try: h = sha1_file(p); h2p[h].append(str(p))
        except: pass
    return h2p

hashes_by_split = {}
for sp in SPLITS:
    paths = list((OUT_ROOT/sp/"real").glob("*")) + list((OUT_ROOT/sp/"fake").glob("*"))
    hashes_by_split[sp] = collect_hashes(paths)

# Cross-split
for i,a in enumerate(SPLITS):
    for b in SPLITS[i+1:]:
        inter = set(hashes_by_split[a]).intersection(hashes_by_split[b])
        print(f"- Cross-split duplicates {a} vs {b}: {len(inter)}")
        for h in list(inter)[:5]:
            print("  ", hashes_by_split[a][h][0])
            print("  ", hashes_by_split[b][h][0])

# Within-split real vs fake
for sp in SPLITS:
    real_hash = collect_hashes((OUT_ROOT/sp/"real").glob("*"))
    fake_hash = collect_hashes((OUT_ROOT/sp/"fake").glob("*"))
    inter = set(real_hash).intersection(fake_hash)
    print(f"- Real↔Fake identical within {sp}: {len(inter)}")
    for h in list(inter)[:5]:
        print("  ", real_hash[h][0])
        print("  ", fake_hash[h][0])

print("\nDone.")


Available images (found):
- train: real=11326 (mode=species) | fakes by tech = {'stargan_tiles': 992, 'copy_move': 446, 'inpaint': 450, 'postproc': 450, 'splicing': 300, 'swap_like': 600}
  layout: {'stargan_tiles': 'recursive', 'copy_move': 'species', 'inpaint': 'species', 'postproc': 'species', 'splicing': 'recursive', 'swap_like': 'species'}
- val: real=2431 (mode=species) | fakes by tech = {'stargan_tiles': 992, 'copy_move': 446, 'inpaint': 450, 'postproc': 450, 'splicing': 300, 'swap_like': 600}
  layout: {'stargan_tiles': 'recursive', 'copy_move': 'species', 'inpaint': 'species', 'postproc': 'species', 'splicing': 'recursive', 'swap_like': 'species'}
- holdout: real=2369 (mode=species) | fakes by tech = {'stargan_tiles': 992, 'copy_move': 449, 'inpaint': 450, 'postproc': 450, 'splicing': 300, 'swap_like': 600}
  layout: {'stargan_tiles': 'recursive', 'copy_move': 'species', 'inpaint': 'species', 'postproc': 'species', 'splicing': 'recursive', 'swap_like': 'species'}
[train] wrote

In [ ]:
import tarfile, datetime, os
from pathlib import Path

base_dir = Path("/content/stargan-v2")
src_dir  = base_dir / "data/combined_balanced"
dst_dir  = Path("/content/drive/MyDrive/stargan-v2/combined_balanced")
dst_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.datetime.now().strftime("%Y%m%d-%H%M")
archive_path = dst_dir / f"afhq_all-{ts}.tar.gz"

# Count files for a rough progress indicator
files = [p for p in src_dir.rglob("*") if p.is_file()]
print("Files to pack:", len(files))

count = 0
with tarfile.open(archive_path, mode="w:gz") as tar:
    tar.dereference = True   # follow symlinks
    for p in files:
        arcname = p.relative_to(base_dir)  # will create "data/afhq_all/..."
        tar.add(p, arcname=str(arcname))
        count += 1
        if count % 1000 == 0:
            print(f"Packed {count}/{len(files)}")

print("Wrote:", archive_path)
print("Size (bytes):", os.path.getsize(archive_path))

Files to pack: 14764
Packed 1000/14764
Packed 2000/14764
Packed 3000/14764
Packed 4000/14764
Packed 5000/14764
Packed 6000/14764
Packed 7000/14764
Packed 8000/14764
Packed 9000/14764
Packed 10000/14764
Packed 11000/14764
Packed 12000/14764
Packed 13000/14764
Packed 14000/14764
Wrote: /content/drive/MyDrive/stargan-v2/combined_balanced/afhq_all-20250824-1622.tar.gz
Size (bytes): 2659149530
